# COMP 469 — Chapter 3 Search, Self-Contained

This notebook contains the Chapter 3 search subset needed for classroom demos. It does not require the full `aima-python` repository.

Included: `Problem`, `Node`, Romania route problem, BFS, DFS, UCS, DLS, IDS, greedy best-first, A*, and RBFS.


In [ ]:
"""
COMP 469 / AIMA Chapter 3 Search - Self-Contained Classroom Subset

This file is intentionally standalone: it uses only the Python standard library.

Included:
- Problem
- Node
- SimpleProblemSolvingAgent
- Graph and Romania map
- breadth_first_tree_search
- depth_first_tree_search
- breadth_first_graph_search
- depth_first_graph_search
- uniform_cost_search
- depth_limited_search
- iterative_deepening_search
- best_first_graph_search
- astar_search
- recursive_best_first_search

Run:
    python aima_ch3_search_standalone.py
"""

from __future__ import annotations

from collections import deque
from dataclasses import dataclass
import heapq
import itertools
import math
from typing import Any, Callable, Dict, Iterable, List, Optional, Tuple


class Problem:
    """Abstract search problem.

    To define a concrete problem, subclass this and implement actions() and result().
    Override goal_test(), path_cost(), and h() when needed.
    """

    def __init__(self, initial: Any, goal: Optional[Any] = None):
        self.initial = initial
        self.goal = goal

    def actions(self, state: Any) -> Iterable[Any]:
        """Return actions available in the given state."""
        raise NotImplementedError

    def result(self, state: Any, action: Any) -> Any:
        """Return the state that results from doing action in state."""
        raise NotImplementedError

    def goal_test(self, state: Any) -> bool:
        """Return True if state is a goal state."""
        if isinstance(self.goal, (list, tuple, set)):
            return state in self.goal
        return state == self.goal

    def path_cost(self, cost_so_far: float, state1: Any, action: Any, state2: Any) -> float:
        """Return total path cost after moving from state1 to state2 by action."""
        return cost_so_far + 1

    def h(self, node: "Node") -> float:
        """Heuristic estimate from node to goal. Default is uninformed."""
        return 0


@dataclass
class Node:
    """A node in a search tree."""

    state: Any
    parent: Optional["Node"] = None
    action: Optional[Any] = None
    path_cost: float = 0
    depth: int = 0

    def __post_init__(self):
        if self.parent is not None:
            self.depth = self.parent.depth + 1

    def __lt__(self, other: "Node") -> bool:
        return self.path_cost < other.path_cost

    def child_node(self, problem: Problem, action: Any) -> "Node":
        next_state = problem.result(self.state, action)
        next_cost = problem.path_cost(self.path_cost, self.state, action, next_state)
        return Node(next_state, self, action, next_cost)

    def expand(self, problem: Problem) -> List["Node"]:
        return [self.child_node(problem, action) for action in problem.actions(self.state)]

    def solution(self) -> List[Any]:
        """Return actions from root to this node."""
        return [node.action for node in self.path()[1:]]

    def path(self) -> List["Node"]:
        """Return nodes from root to this node."""
        node = self
        result = []
        while node is not None:
            result.append(node)
            node = node.parent
        return list(reversed(result))

    def path_states(self) -> List[Any]:
        return [node.state for node in self.path()]


class PriorityQueue:
    """Small priority queue wrapper for search frontiers."""

    def __init__(self, f: Callable[[Node], float]):
        self.f = f
        self.heap: List[Tuple[float, int, Node]] = []
        self.counter = itertools.count()

    def append(self, node: Node) -> None:
        heapq.heappush(self.heap, (self.f(node), next(self.counter), node))

    def pop(self) -> Node:
        return heapq.heappop(self.heap)[2]

    def __len__(self) -> int:
        return len(self.heap)


class Graph:
    """Weighted graph.

    graph_dict format:
        {
            'A': {'B': 5, 'C': 2},
            'B': {'D': 4}
        }
    """

    def __init__(self, graph_dict: Optional[Dict[Any, Dict[Any, float]]] = None, directed: bool = True):
        self.graph_dict: Dict[Any, Dict[Any, float]] = graph_dict or {}
        self.directed = directed
        if not directed:
            self.make_undirected()

    def make_undirected(self) -> None:
        for a in list(self.graph_dict.keys()):
            for b, distance in list(self.graph_dict[a].items()):
                self.connect(b, a, distance)

    def connect(self, a: Any, b: Any, distance: float = 1) -> None:
        self.graph_dict.setdefault(a, {})[b] = distance
        if not self.directed:
            self.graph_dict.setdefault(b, {})[a] = distance

    def get(self, a: Any, b: Optional[Any] = None):
        links = self.graph_dict.setdefault(a, {})
        if b is None:
            return links
        return links.get(b, math.inf)

    def nodes(self) -> List[Any]:
        result = set(self.graph_dict.keys())
        for neighbors in self.graph_dict.values():
            result.update(neighbors.keys())
        return sorted(result)


class GraphProblem(Problem):
    """Route-finding problem on a weighted graph."""

    def __init__(self, initial: Any, goal: Any, graph: Graph, heuristic: Optional[Dict[Any, float]] = None):
        super().__init__(initial, goal)
        self.graph = graph
        self.heuristic = heuristic or {}

    def actions(self, state: Any) -> Iterable[Any]:
        return self.graph.get(state).keys()

    def result(self, state: Any, action: Any) -> Any:
        return action

    def path_cost(self, cost_so_far: float, state1: Any, action: Any, state2: Any) -> float:
        return cost_so_far + self.graph.get(state1, state2)

    def h(self, node: Node) -> float:
        return self.heuristic.get(node.state, 0)


romania_map = Graph(
    {
        "Arad": {"Zerind": 75, "Sibiu": 140, "Timisoara": 118},
        "Bucharest": {"Urziceni": 85, "Pitesti": 101, "Giurgiu": 90, "Fagaras": 211},
        "Craiova": {"Drobeta": 120, "Rimnicu Vilcea": 146, "Pitesti": 138},
        "Drobeta": {"Mehadia": 75},
        "Eforie": {"Hirsova": 86},
        "Fagaras": {"Sibiu": 99},
        "Hirsova": {"Urziceni": 98},
        "Iasi": {"Vaslui": 92, "Neamt": 87},
        "Lugoj": {"Timisoara": 111, "Mehadia": 70},
        "Oradea": {"Zerind": 71, "Sibiu": 151},
        "Pitesti": {"Rimnicu Vilcea": 97},
        "Rimnicu Vilcea": {"Sibiu": 80},
        "Urziceni": {"Vaslui": 142},
    },
    directed=False,
)

# Straight-line distance to Bucharest, commonly used for A* on the Romania map.
romania_sld_to_bucharest = {
    "Arad": 366,
    "Bucharest": 0,
    "Craiova": 160,
    "Drobeta": 242,
    "Eforie": 161,
    "Fagaras": 176,
    "Giurgiu": 77,
    "Hirsova": 151,
    "Iasi": 226,
    "Lugoj": 244,
    "Mehadia": 241,
    "Neamt": 234,
    "Oradea": 380,
    "Pitesti": 100,
    "Rimnicu Vilcea": 193,
    "Sibiu": 253,
    "Timisoara": 329,
    "Urziceni": 80,
    "Vaslui": 199,
    "Zerind": 374,
}


def breadth_first_tree_search(problem: Problem) -> Optional[Node]:
    frontier = deque([Node(problem.initial)])
    while frontier:
        node = frontier.popleft()
        if problem.goal_test(node.state):
            return node
        frontier.extend(node.expand(problem))
    return None


def depth_first_tree_search(problem: Problem, limit: int = 50) -> Optional[Node]:
    frontier = [Node(problem.initial)]
    while frontier:
        node = frontier.pop()
        if problem.goal_test(node.state):
            return node
        if node.depth < limit:
            frontier.extend(reversed(node.expand(problem)))
    return None


def breadth_first_graph_search(problem: Problem) -> Optional[Node]:
    node = Node(problem.initial)
    if problem.goal_test(node.state):
        return node

    frontier = deque([node])
    reached = {problem.initial}

    while frontier:
        node = frontier.popleft()
        for child in node.expand(problem):
            if child.state not in reached:
                if problem.goal_test(child.state):
                    return child
                reached.add(child.state)
                frontier.append(child)
    return None


def depth_first_graph_search(problem: Problem, limit: int = 50) -> Optional[Node]:
    frontier = [Node(problem.initial)]
    reached = set()

    while frontier:
        node = frontier.pop()
        if problem.goal_test(node.state):
            return node
        if node.state not in reached and node.depth < limit:
            reached.add(node.state)
            frontier.extend(reversed(node.expand(problem)))
    return None


def best_first_graph_search(problem: Problem, f: Callable[[Node], float]) -> Optional[Node]:
    node = Node(problem.initial)
    frontier = PriorityQueue(f)
    frontier.append(node)

    best_score = {node.state: f(node)}

    while frontier:
        node = frontier.pop()

        if problem.goal_test(node.state):
            return node

        for child in node.expand(problem):
            score = f(child)
            if child.state not in best_score or score < best_score[child.state]:
                best_score[child.state] = score
                frontier.append(child)

    return None


def uniform_cost_search(problem: Problem) -> Optional[Node]:
    return best_first_graph_search(problem, lambda node: node.path_cost)


def astar_search(problem: Problem, h: Optional[Callable[[Node], float]] = None) -> Optional[Node]:
    heuristic = h or problem.h
    return best_first_graph_search(problem, lambda node: node.path_cost + heuristic(node))


def depth_limited_search(problem: Problem, limit: int = 50):
    """Return a solution Node, None for failure, or the string 'cutoff'."""

    def recursive_dls(node: Node, problem: Problem, limit_remaining: int):
        if problem.goal_test(node.state):
            return node
        if limit_remaining == 0:
            return "cutoff"

        cutoff_occurred = False
        for child in node.expand(problem):
            # Avoid trivial cycles on graph problems.
            if child.state in node.path_states():
                continue

            result = recursive_dls(child, problem, limit_remaining - 1)
            if result == "cutoff":
                cutoff_occurred = True
            elif result is not None:
                return result

        return "cutoff" if cutoff_occurred else None

    return recursive_dls(Node(problem.initial), problem, limit)


def iterative_deepening_search(problem: Problem, max_depth: int = 50) -> Optional[Node]:
    for depth in range(max_depth + 1):
        result = depth_limited_search(problem, depth)
        if result != "cutoff":
            return result
    return None


def recursive_best_first_search(problem: Problem, h: Optional[Callable[[Node], float]] = None) -> Optional[Node]:
    heuristic = h or problem.h

    def rbfs(node: Node, f_limit: float) -> Tuple[Optional[Node], float]:
        if problem.goal_test(node.state):
            return node, 0

        successors = []
        current_path_states = set(node.path_states())

        for child in node.expand(problem):
            if child.state not in current_path_states:
                child.f = max(child.path_cost + heuristic(child), getattr(node, "f", 0))
                successors.append(child)

        if not successors:
            return None, math.inf

        while True:
            successors.sort(key=lambda n: n.f)
            best = successors[0]

            if best.f > f_limit:
                return None, best.f

            alternative = successors[1].f if len(successors) > 1 else math.inf
            result, best.f = rbfs(best, min(f_limit, alternative))

            if result is not None:
                return result, best.f

    start = Node(problem.initial)
    start.f = heuristic(start)
    result, _ = rbfs(start, math.inf)
    return result


class SimpleProblemSolvingAgent:
    """A tiny configurable simple problem-solving agent.

    This is a classroom-friendly version:
    - The user provides a problem_factory that creates a Problem from a state and goal.
    - The user provides a search algorithm.
    - The agent computes a sequence of actions, then returns one action per call.
    """

    def __init__(
        self,
        initial_state: Any,
        goal: Any,
        problem_factory: Callable[[Any, Any], Problem],
        search_algorithm: Callable[[Problem], Optional[Node]] = breadth_first_graph_search,
    ):
        self.state = initial_state
        self.goal = goal
        self.problem_factory = problem_factory
        self.search_algorithm = search_algorithm
        self.action_sequence: List[Any] = []

    def update_state(self, percept: Any) -> Any:
        self.state = percept
        return self.state

    def formulate_goal(self, state: Any) -> Any:
        return self.goal

    def formulate_problem(self, state: Any, goal: Any) -> Problem:
        return self.problem_factory(state, goal)

    def __call__(self, percept: Any) -> Optional[Any]:
        state = self.update_state(percept)

        if not self.action_sequence:
            goal = self.formulate_goal(state)
            problem = self.formulate_problem(state, goal)
            solution_node = self.search_algorithm(problem)
            self.action_sequence = solution_node.solution() if solution_node else []

        if self.action_sequence:
            return self.action_sequence.pop(0)

        return None


def print_solution(name: str, node: Optional[Node]) -> None:
    if node is None:
        print(f"{name}: no solution")
        return

    print(f"{name}:")
    print(f"  Path: {' -> '.join(map(str, node.path_states()))}")
    print(f"  Actions: {node.solution()}")
    print(f"  Cost: {node.path_cost}")
    print(f"  Depth: {node.depth}")
    print()


def demo_romania() -> None:
    problem = GraphProblem("Arad", "Bucharest", romania_map, romania_sld_to_bucharest)

    algorithms = [
        ("Breadth-first graph search", breadth_first_graph_search),
        ("Depth-first graph search", depth_first_graph_search),
        ("Uniform-cost search", uniform_cost_search),
        ("Depth-limited search, limit=3", lambda p: depth_limited_search(p, limit=3)),
        ("Iterative-deepening search", iterative_deepening_search),
        ("Greedy best-first search", lambda p: best_first_graph_search(p, p.h)),
        ("A* search", astar_search),
        ("Recursive best-first search", recursive_best_first_search),
    ]

    for name, algorithm in algorithms:
        print_solution(name, algorithm(problem))




## Demo: Romania route problem


In [ ]:
problem = GraphProblem("Arad", "Bucharest", romania_map, romania_sld_to_bucharest)

algorithms = [
    ("Breadth-first graph search", breadth_first_graph_search),
    ("Depth-first graph search", depth_first_graph_search),
    ("Uniform-cost search", uniform_cost_search),
    ("Depth-limited search, limit=3", lambda p: depth_limited_search(p, limit=3)),
    ("Iterative-deepening search", iterative_deepening_search),
    ("Greedy best-first search", lambda p: best_first_graph_search(p, p.h)),
    ("A* search", astar_search),
    ("Recursive best-first search", recursive_best_first_search),
]

for name, algorithm in algorithms:
    print_solution(name, algorithm(problem))


## Demo: Simple problem-solving agent


In [ ]:
def romania_problem_factory(state, goal):
    return GraphProblem(state, goal, romania_map, romania_sld_to_bucharest)

agent = SimpleProblemSolvingAgent(
    initial_state="Arad",
    goal="Bucharest",
    problem_factory=romania_problem_factory,
    search_algorithm=astar_search,
)

current_city = "Arad"
for step in range(5):
    action = agent(current_city)
    print(f'Step {step + 1}: at {current_city}, action = {action}')
    if action is None:
        break
    current_city = action
